# Exploratory raw-waveform reanalysis

This independent implementation is not Doric's proprietary algorithm. Validate it with lifetime standards before scientific use.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from fluopulse_analysis import (
    fit_irf_convolved_double_exponential,
    fit_irf_convolved_single_exponential,
    moment_lifetime_ns,
    plot_waveform_fit,
    read_doric,
)

In [ ]:
DORIC_FILE = Path.home() / "Documents" / "SC81_260923" / "260923_SC81_run1_0000.doric"
WAVEFORM_INDEX = 1000
FIT_WINDOW_NS = (0.0, 40.0)
if not DORIC_FILE.is_file():
    raise FileNotFoundError(DORIC_FILE)
recording = read_doric(DORIC_FILE, extract_digital=False)

In [ ]:
batch = recording.load_waveforms([WAVEFORM_INDEX])
waveform = batch.values[0]
fit = fit_irf_convolved_single_exponential(batch.waveform_time_ns, waveform, recording.irf_values, fit_window_ns=FIT_WINDOW_NS)
moment = moment_lifetime_ns(batch.waveform_time_ns, waveform, recording.irf_values, window_ns=FIT_WINDOW_NS)
print(f"Vendor Tau01: {recording.tau_ns[WAVEFORM_INDEX]:.6f} ns")
print(f"Independent convolved fit: {fit.tau_ns:.6f} ns (R2={fit.r_square:.6f})")
print(f"Independent moment estimate: {moment:.6f} ns")

In [ ]:
figure = plot_waveform_fit(batch.waveform_time_ns, waveform, fit)
plt.show()

In [ ]:
double_fit = fit_irf_convolved_double_exponential(
    batch.waveform_time_ns, waveform, recording.irf_values, fit_window_ns=FIT_WINDOW_NS
)
print(f"Double fit: {double_fit.tau_short_ns:.4f} ns and {double_fit.tau_long_ns:.4f} ns")
print(f"Long-component fraction: {double_fit.long_fraction:.4f}")
print(f"R2: {double_fit.r_square:.6f}")

Repeat across calibration standards, waveform indices, signal amplitudes, and fit windows. Agreement on one waveform is not validation.